# Data cleaning

Inspect raw reviews, remove duplicate review-rating pairs, reconcile conflicting ratings, and normalize and tokenize the review text.

Run from the repository root.

In [ ]:

import os
import sys

# Set environment variables
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# Create a SparkSession
spark = (
    SparkSession.builder
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)
# Initialize
ss = spark

sc = ss.sparkContext

ss.sparkContext.setLogLevel("ERROR")
# from pyspark.ml.feature import CountVectorizer, IDF, Tokenizer


### Full Steps of Data Cleaning
- Delete duplicate data
- Resolve scoring conflicts
- Remove missing values
- Handle invalid values (non-existent)
- Data standardization
- Stemming
- String tokenization
- Tokenize single‑quoted concatenated terms
- Remove stop words
- split dataset


In [ ]:
# Convert to a DataFrame
data = ss.read.csv(path='drugsComTrain_raw.csv',
                        sep=',',
                        encoding='UTF-8',
                        comment=None,
                        header=True,
                        inferSchema=True,
                        quote='"',        # Explicitly specify double quotes as the text delimiter
                        escape='"',       # Escape internal quotes by doubling them
                        multiLine=True    # Allow a field to span multiple lines
                        )


data.select('usefulCount').sort('usefulCount') # Data looks valid; minimum value = 0
data.show(20, truncate=100)
 

In [ ]:
# Keep only review and rating 

df_data = data.select('review','rating').toDF('review','rating')
df_data.count()



In [ ]:
# # Remove duplicates
df_data = df_data.distinct()
# df_data.count()
df_data.show(20, truncate=100)

In [ ]:
# Label conflicts: same review with different ratings
# Count reviews with more than one rating
from pyspark.sql.functions import col, trim
conflict_reviews = (
    df_data.groupBy("review")
    .agg(
        f.countDistinct("rating").alias("different_ratings")
    )
    .filter(col("different_ratings") > 1)
)

conflict_records = df_data.join(
    conflict_reviews.select("review"),
    on="review",
    how="inner"
)

print("Rows with conflicting ratings:", conflict_records.count())

conflict_reviews.sort("different_ratings", ascending=False).show(20)
# The maximum number of duplicates is 6, and the minimum is 2




In [ ]:
from pyspark.sql import functions as f
# For conflicting reviews, merge into one row and round the mean rating to an integer
conflict_rating = (
    df_data.join(conflict_reviews.select("review"), on="review", how="inner")
    .groupBy("review")
    .agg(
        f.round(f.avg("rating")).cast("int").alias("rating")
    )
)
# conflict_rating.show(5)
# conflict_rating.count()

# 3. Select normal data without conflicts
normal_data = df_data.join(
    conflict_reviews.select("review"),
    on="review",
    how="leftanti"   # leftanti: Keep only left-side rows with no match (non-conflicting rows)
)
# 4. Combine normal data with the merged conflicting rows
df_clean = normal_data.unionByName(conflict_rating)
# df_clean.count()


In [ ]:
# Handle missing values
df_clean = df_clean.filter(
    (col("review").isNotNull()) & (trim(col("review")) != "") &
    (col("rating").isNotNull()) & (trim(col("rating")) != "")
)
df_clean.count()

In [ ]:
# Check whether the data is valid
df_clean = df_clean.filter(
    col("rating").between(1, 10))

df_clean.count() 

In [ ]:
# Normalize data
import html
import re
import unicodedata
from pyspark.sql.types import StringType

def normalize_review(text):
    if text is None:
        return None
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    # Remove common HTML tags
    text = re.sub(r"</?[A-Za-z][^>]*>", " ", text)
    # Normalize curly quotes
    text = text.replace("’", "'").replace("‘", "'")
    # Collapse spaces, newlines, and tabs
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Register the UDF on a separate line
normalize_review = f.udf(normalize_review, returnType=StringType())

df_clean = df_clean.withColumn(
    "review",
    normalize_review(col("review"))
)

# Lowercase; keep letters, numbers, whitespace, and ASCII apostrophes
# Keep apostrophes to avoid breaking contractions such as "don't"
df_norm = df_clean.withColumn(
    "review",
    trim(
        f.regexp_replace(
            f.regexp_replace(
                f.lower(col("review")),
                r"[^\p{L}\p{N}\s']",
                " "
            ),
            r"\s+",
            " "
        )
    )
)
df_norm.show(20, truncate=100)

In [ ]:
# Stemming
from nltk.stem import PorterStemmer

ps = PorterStemmer()

def stemming(data_str):
    # input is string
    cleaned_str = ''
    text = data_str.split()

    for word in text:
        cleaned_str+= ps.stem(word) + ' '

    return cleaned_str

#define udf
sent_to_stem_words_udf = f.udf(lambda x: str(stemming(x)), StringType())


df_stem = df_norm.select('review', 'rating', sent_to_stem_words_udf(df_norm.review).alias('stemmed_review'))
df_stem.show(10, truncate=100)

In [ ]:
# Tokenization
from pyspark.ml.feature import  Tokenizer
tokenizer = Tokenizer(inputCol="stemmed_review", outputCol="words") # Tokenizer splits each document into words on whitespace
wordsData = tokenizer.transform(df_stem)
wordsData.show(20, truncate=100)

In [ ]:
# Inspect words containing apostrophes

result = (
    wordsData
    .select(
        "review",
        "rating",
        f.explode("words").alias("token")
    )
    .filter(col("token").contains("'"))
)

# View all unique tokens containing apostrophes
result.select("token").distinct().show(10, truncate=False)



In [ ]:
from pyspark.sql.types import ArrayType
# Expand contractions containing apostrophes
SPECIAL_FORMS = {
    "can't": ["can", "not"],
    "cannot": ["can", "not"],
    "won't": ["will", "not"],
    "shan't": ["shall", "not"],
    # The auxiliary in "ain't" depends on context; retain only the negation
    "ain't": ["not"]
}

# Expansion function
def normalize_tokens(tokens):
    if tokens is None:
        return []

    output = []

    for token in tokens:
        if not token:
            continue

        word = token.lower().replace("’", "'").replace("‘", "'")

        # Remove surrounding quotes, e.g. 'dopey' → dopey
        word = word.strip("'\"")

        if not word:
            continue

        # Special negative forms
        if word in SPECIAL_FORMS:
            output.extend(SPECIAL_FORMS[word])

        # don't → do not；wasn't → was not
        elif word.endswith("n't"):
            base = word[:-3]
            if base:
                output.append(base)
            output.append("not")

        # i'm → i am
        elif word.endswith("'m"):
            output.extend([word[:-2], "am"])

        # you're → you are
        elif word.endswith("'re"):
            output.extend([word[:-3], "are"])

        # i've → i have
        elif word.endswith("'ve"):
            output.extend([word[:-3], "have"])

        # you'll → you will
        elif word.endswith("'ll"):
            output.extend([word[:-3], "will"])

        # "'d" may mean "had" or "would"; retain the stem for now
        elif word.endswith("'d"):
            output.append(word[:-2])

        # "'s" may be possessive, "is", or "has"; retain the stem for now
        # tourette's → tourette; it's → it
        elif word.endswith("'s"):
            output.append(word[:-2])

        else:
            # Do not remove other words containing apostrophes
            output.append(word)

    return [word for word in output if word]

# This returns array<string>
normalize_tokens = f.udf(normalize_tokens, returnType=ArrayType(StringType()))

wordsData = wordsData.withColumn(
    "normalized_words",
    normalize_tokens(col("words"))
)

wordsData.select(
    "words", "normalized_words"
).show(20, truncate=100)


In [ ]:
# Stop-word handling
from pyspark.ml.feature import StopWordsRemover

# Negation words to preserve
protected_words = {
    "not", "no", "nor", "never", "neither", "without"
}

default_stopwords = StopWordsRemover.loadDefaultStopWords("english")

custom_stopwords = [
    word for word in default_stopwords
    if word.lower() not in protected_words
]

remover = StopWordsRemover(
    inputCol="normalized_words",
    outputCol="filtered_words",
    stopWords=custom_stopwords,
    caseSensitive=False
)

cleanData = remover.transform(wordsData)

cleanData.select(
    "normalized_words",
    "filtered_words",
).show(20, truncate=100)



In [ ]:
# 80% training set, 20% test set
train_data, test_data = cleanData.randomSplit(
    [0.8, 0.2],
    seed=42
)

# Cache for convenient reuse later
train_data = train_data.cache()
test_data = test_data.cache()

print("tarin set number:", train_data.count())
print("test set number:", test_data.count())
